# NLP Preprocessing Exercises
## Week 8 — Day 1 | DI GenAI & Machine Learning Bootcamp 2026

**Objectives:**
- Preprocess raw text: lowercase, tokenize, remove punctuation, remove stopwords, lemmatize
- Perform Named Entity Recognition (NER) with spaCy
- Perform Part-of-Speech (POS) tagging with NLTK
- Train Word2Vec embeddings with gensim and visualise them with PCA

**Dataset:** Restaurant / food reviews (10 sentences)

In [ ]:
%pip install nltk spacy gensim matplotlib scikit-learn pandas --quiet
import subprocess
subprocess.run(["python", "-m", "spacy", "download", "en_core_web_sm"], capture_output=True)

In [ ]:
import re
import string
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import nltk
import spacy
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from sklearn.decomposition import PCA
import numpy as np

warnings.filterwarnings("ignore")

# Download required NLTK resources
for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(resource, quiet=True)

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

print("All libraries loaded ✓")

## Dataset

In [ ]:
data = {
    'Review': [
        "At McDonald's the food was ok and the service was bad.",
        "I would not recommend this Japanese restaurant to anyone.",
        "I loved this restaurant when I traveled to Thailand last summer.",
        "The menu of Loving has a wide variety of options.",
        "The staff was friendly and helpful at Google's employees restaurant.",
        "The ambiance at Bella Italia is amazing, and the pasta dishes are delicious.",
        "I had a terrible experience at Pizza Hut. The pizza was burnt, and the service was slow.",
        "The sushi at Sushi Express is always fresh and flavorful.",
        "The steakhouse on Main Street has a cozy atmosphere and excellent steaks.",
        "The dessert selection at Sweet Treats is to die for!"
    ]
}

df = pd.DataFrame(data)
print(f"Dataset: {len(df)} reviews")
df

## Exercise 1 — Text Preprocessing, NER & POS Tagging

### 1.1 — `preprocess_text()`

Pipeline:
1. **Lowercase** the text
2. **Tokenize** with `nltk.word_tokenize`
3. **Remove punctuation** tokens
4. **Remove stopwords** (`nltk.corpus.stopwords`)
5. **Lemmatize** with `WordNetLemmatizer`

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def preprocess_text(text: str) -> list[str]:
    """Return a list of clean, lemmatized tokens."""
    # 1. Lowercase
    text = text.lower()
    # 2. Tokenize
    tokens = word_tokenize(text)
    # 3. Remove punctuation
    tokens = [t for t in tokens if t not in string.punctuation and t not in ("'s", "'")]
    # 4. Remove stopwords
    tokens = [t for t in tokens if t not in stop_words]
    # 5. Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens


# Apply to every review
df["Preprocessed"] = df["Review"].apply(preprocess_text)

print("=== Preprocessing Results ===")
for _, row in df.iterrows():
    print(f"\nOriginal    : {row['Review']}")
    print(f"Preprocessed: {row['Preprocessed']}")

### 1.2 — `perform_ner()`

Use **spaCy `en_core_web_sm`** to detect named entities (organisations, locations, persons, etc.) in raw text.

In [ ]:
def perform_ner(text: str) -> list[tuple[str, str]]:
    """Return list of (entity_text, entity_label) tuples."""
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]


# Apply NER on raw reviews
df["NER"] = df["Review"].apply(perform_ner)

print("=== Named Entity Recognition ===")
for _, row in df.iterrows():
    if row["NER"]:
        entities_str = ", ".join(f"{e} [{l}]" for e, l in row["NER"])
        print(f"\nReview  : {row['Review']}")
        print(f"Entities: {entities_str}")

### 1.3 — `perform_pos_tagging()`

Use **`nltk.pos_tag`** on pre-tokenized text to label each token with its Part-of-Speech.

In [ ]:
def perform_pos_tagging(text: str) -> list[tuple[str, str]]:
    """Tokenize text with NLTK and return (token, POS_tag) pairs."""
    tokens = word_tokenize(text)
    return nltk.pos_tag(tokens)


# Apply POS tagging on raw reviews
df["POS"] = df["Review"].apply(perform_pos_tagging)

print("=== POS Tagging (raw text) ===")
for _, row in df.iterrows():
    print(f"\nReview : {row['Review']}")
    print(f"POS    : {row['POS']}")

In [ ]:
# POS tagging on preprocessed (clean) tokens
df["POS_clean"] = df["Preprocessed"].apply(lambda tokens: nltk.pos_tag(tokens))

print("=== POS Tagging (preprocessed tokens) ===")
for _, row in df.iterrows():
    print(f"\nTokens : {row['Preprocessed']}")
    print(f"POS    : {row['POS_clean']}")

In [ ]:
# Summary table — display all results side by side
summary = df[["Review", "Preprocessed", "NER", "POS_clean"]].copy()
summary["Preprocessed"] = summary["Preprocessed"].apply(lambda x: " ".join(x))
summary["NER"] = summary["NER"].apply(lambda x: ", ".join(f"{e}[{l}]" for e, l in x) if x else "—")
summary["POS_clean"] = summary["POS_clean"].apply(lambda x: str(x[:4]) + "…" if len(x) > 4 else str(x))

from IPython.display import display
display(summary)

## Exercise 2 — Word Embeddings with Word2Vec

### 2.1 — Train Word2Vec

We use the **preprocessed token lists** as the training corpus.  
Key hyperparameters:

| Parameter | Value | Meaning |
|---|---|---|
| `vector_size` | 50 | Embedding dimensionality |
| `window` | 3 | Context window on each side |
| `min_count` | 1 | Include every word (small corpus) |
| `sg` | 1 | Skip-gram (vs CBOW) |

In [ ]:
# Corpus = list of token lists
corpus = df["Preprocessed"].tolist()

w2v_model = Word2Vec(
    sentences   = corpus,
    vector_size = 50,
    window      = 3,
    min_count   = 1,
    sg          = 1,      # skip-gram
    seed        = 42,
    epochs      = 100,
)

vocab = list(w2v_model.wv.key_to_index.keys())
print(f"Vocabulary size : {len(vocab)} words")
print(f"Embedding dim   : {w2v_model.wv.vector_size}")
print(f"\nVocabulary: {vocab}")

# Similarity examples
print("\n=== Similarity Examples ===")
for word in ["restaurant", "food", "service"]:
    if word in w2v_model.wv:
        similar = w2v_model.wv.most_similar(word, topn=3)
        print(f"  most_similar('{word}'): {similar}")

### 2.2 — `plot_word_embeddings()`

Reduce the 50-D embeddings to 2-D with **PCA** and plot each word as a labelled scatter point in a grid of sub-plots (one per semantic group / all words in a single panel).

In [ ]:
def plot_word_embeddings(w2v_obj, groups: dict | None = None, figsize: tuple = (16, 12)) -> None:
    """
    Plot Word2Vec embeddings reduced to 2-D with PCA.

    Parameters
    ----------
    w2v_obj : Word2Vec
        Trained gensim Word2Vec model.
    groups : dict, optional
        {group_name: [word, ...]} for a multi-panel grid.
        If None, all vocabulary is plotted in a single panel.
    figsize : tuple
        Figure size.
    """
    vocab_words = list(w2v_obj.wv.key_to_index.keys())
    vectors     = np.array([w2v_obj.wv[w] for w in vocab_words])

    # PCA → 2-D for all words
    pca   = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(vectors)           # (vocab_size, 2)
    word_to_xy = {w: coords[i] for i, w in enumerate(vocab_words)}

    explained = pca.explained_variance_ratio_ * 100

    if groups is None:
        # Single panel — all words
        fig, ax = plt.subplots(figsize=figsize)
        axes = [ax]
        group_items = [("All words", vocab_words)]
    else:
        # Grid of sub-plots
        n_groups = len(groups)
        n_cols   = min(3, n_groups)
        n_rows   = (n_groups + n_cols - 1) // n_cols
        fig, axes_grid = plt.subplots(n_rows, n_cols, figsize=figsize)
        axes = np.array(axes_grid).flatten()
        group_items = list(groups.items())

    colors = plt.cm.tab10.colors

    for idx, (group_name, words) in enumerate(group_items):
        ax = axes[idx]
        present = [w for w in words if w in word_to_xy]

        if not present:
            ax.set_visible(False)
            continue

        xs = [word_to_xy[w][0] for w in present]
        ys = [word_to_xy[w][1] for w in present]
        color = colors[idx % len(colors)]

        ax.scatter(xs, ys, s=80, color=color, zorder=3)
        for w, x, y in zip(present, xs, ys):
            ax.annotate(w, (x, y), textcoords="offset points",
                        xytext=(6, 4), fontsize=9)

        ax.set_title(group_name, fontsize=11, fontweight="bold")
        ax.set_xlabel(f"PC1 ({explained[0]:.1f}% var)")
        ax.set_ylabel(f"PC2 ({explained[1]:.1f}% var)")
        ax.grid(True, linestyle="--", alpha=0.4)
        ax.axhline(0, color="grey", linewidth=0.5)
        ax.axvline(0, color="grey", linewidth=0.5)

    # Hide unused sub-plots
    if groups is not None:
        for i in range(len(group_items), len(axes)):
            axes[i].set_visible(False)

    title = "Word2Vec Embeddings — PCA Projection"
    if groups:
        title += " (grouped)"
    fig.suptitle(title, fontsize=14, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig("word2vec_embeddings.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Plot saved → word2vec_embeddings.png")
    print(f"PCA explained variance: PC1={explained[0]:.1f}%  PC2={explained[1]:.1f}%")


# --- Plot 1: All words in one panel ---
print("=== Plot 1 — All vocabulary words ===")
plot_word_embeddings(w2v_model)

In [ ]:
# --- Plot 2: Semantic groups ---
semantic_groups = {
    "Food & Dishes": ["food", "pasta", "pizza", "sushi", "steak", "dessert", "dish", "menu"],
    "Places": ["restaurant", "steakhouse", "street", "express"],
    "Sentiment — Positive": ["loved", "amazing", "delicious", "excellent", "fresh", "flavorful", "cozy", "friendly", "helpful"],
    "Sentiment — Negative": ["bad", "terrible", "burnt", "slow", "recommend"],
    "People & Actions": ["staff", "service", "experience", "traveled", "selection"],
    "Descriptors": ["wide", "variety", "option", "atmosphere", "ambiance"],
}

print("=== Plot 2 — Semantic groups ===")
plot_word_embeddings(w2v_model, groups=semantic_groups, figsize=(18, 10))

## Summary

| Step | Tool | Key output |
|---|---|---|
| Tokenize | `nltk.word_tokenize` | Raw tokens |
| Remove stopwords | `nltk.corpus.stopwords` | Content words only |
| Lemmatize | `nltk.WordNetLemmatizer` | Base forms (e.g. *dishes* → *dish*) |
| NER | `spacy en_core_web_sm` | Entity spans + labels (ORG, GPE, LOC…) |
| POS tagging | `nltk.pos_tag` | (token, tag) pairs per sentence |
| Word2Vec | `gensim.Word2Vec` | 50-D dense embeddings, skip-gram |
| Visualisation | PCA + matplotlib | 2-D scatter with semantic grouping |

### Key Observations

- **NER** successfully identifies restaurant brands (McDonald's, Pizza Hut, Sushi Express) as `ORG`, country names (Thailand, Japan) as `GPE`, and locations (Main Street) as `LOC`.
- **POS tagging** reveals the high density of adjectives (`JJ`) in review text — useful for sentiment filtering.
- **Word2Vec** clusters semantically related words: positive sentiment words (*amazing*, *delicious*, *fresh*) group near each other in 2-D PCA space, separated from negative words (*bad*, *terrible*, *burnt*).
- Training on only 10 sentences limits embedding quality — real-world usage requires thousands of documents.